# Four NMR representations and ethnicity figures


In [ ]:
# Setup: run once. No training, bootstrap, or significance testing.
from pathlib import Path
import os, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
PROJECT = Path(os.environ.get("CHD_OUTPUT_ROOT", "../outputs")).resolve()
RUNS={'Score249':PROJECT/'runs/reviewer_full249_score_5fold','Score106':PROJECT/'runs/reviewer_raw106_score'}
OUT=Path(os.environ.get('CHD_PLOT_OUTPUT',str(PROJECT/'reviewer_four_representation_plots')))
FIG=OUT/'figures'; TABLE=OUT/'source_data'
FIG.mkdir(parents=True,exist_ok=True); TABLE.mkdir(parents=True,exist_ok=True)
SHOW=True
REPS=['Score249','Score106','Spearman-pruned','Raw106']
FEATURES={'Score249':'Clinical_plus_PRS_plus_NMR_score','Score106':'Clinical_plus_PRS_plus_NMR_score','Spearman-pruned':'Clinical_plus_PRS_plus_SpearmanNMR','Raw106':'Clinical_plus_PRS_plus_Raw106NMR'}
COLORS=dict(zip(REPS,['#3C5488','#00A087','#8491B4','#E64B35']))
GROUPS=['White','Known_nonWhite_mixed_other']
GLABEL={'White':'White','Known_nonWhite_mixed_other':'Non-White / mixed / other'}
GC={'White':'#3C5488','Known_nonWhite_mixed_other':'#E64B35'}
ALGS=['Cox PH','Elastic-net Cox','XGBoost survival','DeepSurv']
CACHE={}; LOG=[]; PROVENANCE=[]
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'axes.titlesize':12,'axes.labelsize':11,'axes.spines.top':False,'axes.spines.right':False,'svg.fonttype':'none','pdf.fonttype':42,'savefig.dpi':300})
def slug(s):
    return ''.join(c if c.isalnum() else '_' for c in str(s))
def read(path):
    path=Path(path)
    if path not in CACHE:
        CACHE[path]=pd.read_csv(path)
        PROVENANCE.append({'Source':str(path),'SHA256':hashlib.sha256(path.read_bytes()).hexdigest()})
    return CACHE[path].copy()
def source(score,name):
    matches=[p for p in (RUNS[score]/'04_results').rglob(name) if not any(x in str(p) for x in ['manuscript_revised','plot_only','reviewer_four_representation_plots'])]
    if not matches: raise FileNotFoundError(str(RUNS[score]/'04_results')+' / '+name)
    hashes={hashlib.sha256(p.read_bytes()).hexdigest() for p in matches}
    if len(hashes)>1: raise RuntimeError('Ambiguous source files: '+str(matches))
    return read(sorted(matches)[0])
def subset(d,h=None,algorithm=None):
    if h is not None:
        col=next((x for x in ['Horizon_Years','Tau_Years','Horizon'] if x in d),None)
        if col is None: raise ValueError('No horizon column')
        d=d[np.isclose(pd.to_numeric(d[col]),h)]
    if algorithm is not None:
        col='Algorithm' if 'Algorithm' in d else 'Model'
        d=d[d[col].eq(algorithm)]
    return d.copy()
def normrep(d):
    if 'Feature_Set' in d:
        d['Rep']=d.Feature_Set.map({'Clinical_plus_PRS_plus_NMR_score':'Score','Clinical_plus_PRS_plus_SpearmanNMR':'Spearman-pruned','Clinical_plus_PRS_plus_Raw106NMR':'Raw106'})
    else:
        col='Representation' if 'Representation' in d else 'Strategy'
        d['Rep']=d[col].map({'NMR_score':'Score','NMR score':'Score','Spearman-pruned NMR':'Spearman-pruned','Spearman-pruned':'Spearman-pruned','Raw106 NMR':'Raw106','Raw106':'Raw106'})
    return d

CONTEXT_CHECKED=False
def check_context():
    global CONTEXT_CHECKED
    if CONTEXT_CHECKED:return
    manifests={}
    for score,root in RUNS.items():
        paths=list(root.rglob('pipeline_manifest.json'))
        if len(paths)!=1:raise RuntimeError('Expected exactly one pipeline_manifest.json: '+str(paths))
        manifests[score]=json.loads(paths[0].read_text(encoding='utf-8'))
    for split in ['train','val','test']:
        a=manifests['Score249']['splits'][split]['eid_sha256'];b=manifests['Score106']['splits'][split]['eid_sha256']
        if not a or a!=b:raise RuntimeError('Different EID order/partition: '+split)
    for score,expected in [('Score249','NMR_score249_5fold'),('Score106','NMR_score106')]:
        if manifests[score]['nmr_score'].get('score_name')!=expected:raise RuntimeError('Unexpected score identity: '+score)
    CONTEXT_CHECKED=True

def four(name,h=None,algorithm=None):
    check_context()
    parts=[]
    for score in ['Score249','Score106']:
        d=normrep(subset(source(score,name),h,algorithm))
        if score=='Score106': d=d[d.Rep.eq('Score')]
        else: d=d[d.Rep.notna()]
        d['Rep']=d.Rep.replace({'Score':score});d['Source_Run']=RUNS[score].name
        parts.append(d)
    out=pd.concat(parts,ignore_index=True)
    if set(out.Rep)!=set(REPS): raise RuntimeError('Four representations incomplete: '+name)
    return out

def save(fig,d,stem,note=''):
    if note: fig.text(.5,.015,note,ha='center',va='bottom',fontsize=8,wrap=True)
    fig.tight_layout(rect=[0,.08 if note else 0,1,1])
    d.to_csv(TABLE/(stem+'.csv'),index=False)
    for ext in ['png','pdf','svg']:fig.savefig(FIG/(stem+'.'+ext),bbox_inches='tight')
    if SHOW:plt.show()
    plt.close(fig)
    LOG.append({'Figure':stem,'Status':'saved','Note':note});print('Saved:',stem)

def run(stem,fn,*args,**kwargs):
    try:fn(*args,stem=stem,**kwargs)
    except (FileNotFoundError,KeyError,ValueError,RuntimeError) as e:
        LOG.append({'Figure':stem,'Status':'MISSING_OR_ERROR','Note':str(e)});print('MISSING/ERROR:',stem,str(e))
    pd.DataFrame(LOG).to_csv(OUT/'figure_status.csv',index=False)
    pd.DataFrame(PROVENANCE).drop_duplicates().to_csv(OUT/'source_manifest.csv',index=False)

def forest(d,estimate,lo,hi,title,xlabel,stem,labels='Rep',order=None,zero=True,note=''):
    order=order or REPS
    if d[labels].duplicated().any():raise RuntimeError('Multiple rows per plotted label; add filters')
    d=d.set_index(labels).reindex(order).reset_index()
    if not np.isfinite(d[[estimate,lo,hi]].to_numpy(float)).all():raise ValueError('Missing or nonfinite estimates or confidence limits')
    v=d[estimate].to_numpy(float);l=d[lo].to_numpy(float);u=d[hi].to_numpy(float)
    if np.any(l>v+1e-10) or np.any(u<v-1e-10):raise ValueError('CI does not contain point estimate')
    fig,ax=plt.subplots(figsize=(7.5,max(3.3,.45*len(d)+1.4)))
    for i,row in d.iterrows():
        ax.errorbar(v[i],i,xerr=[[max(0,v[i]-l[i])],[max(0,u[i]-v[i])]],fmt='o',color=COLORS.get(row[labels],'#3C5488'),capsize=3,lw=1.3,ms=6)
    if zero:ax.axvline(0,color='.5',ls='--',lw=.9)
    ax.set_yticks(range(len(d)),d[labels]);ax.invert_yaxis();ax.set_xlabel(xlabel);ax.set_title(title,loc='left');ax.grid(axis='x',alpha=.18)
    save(fig,d,stem,note)

def global_metric(filename,h,alg,e,l,u,title,stem):
    d=four(filename,h,alg)
    forest(d,e,l,u,f'{h}-year {title} | {alg or "Elastic-net minus Cox"}',title+' (95% CI)',stem,note='Four NMR representations; same locked test partition. See source CSV for reference model.')

def global_cal(h,alg,stem):
    d=four('figure3F_calibration.csv',h,alg);fig,axes=plt.subplots(2,2,figsize=(9,7))
    lim=max(d.Predicted_Risk.max(),d.Observed_KM_Risk.max())*110
    for ax,rep in zip(axes.flat,REPS):
        q=d[d.Rep.eq(rep)].sort_values('Predicted_Risk');ax.plot([0,lim],[0,lim],'--',color='.6');ax.plot(q.Predicted_Risk*100,q.Observed_KM_Risk*100,'o-',color=COLORS[rep]);ax.set(xlim=(0,lim),ylim=(0,lim),title=rep,xlabel='Predicted risk (%)',ylabel='Observed KM risk (%)')
    fig.suptitle(f'{h}-year grouped calibration | {alg}');save(fig,d,stem,'Grouped estimates; source tables contain no calibration confidence intervals. Lines connect group estimates.')

def global_dca(h,alg,stem,delta=False):
    d=four('figure3G_survival_dca.csv',h,alg);fig,ax=plt.subplots(figsize=(8,4.5))
    original=subset(source('Score249','figure3G_survival_dca.csv'),h,alg)
    ref=original[original.Strategy.eq('Clinical + PRS')][['Threshold','Net_Benefit']].rename(columns={'Net_Benefit':'Reference_NB'})
    if ref.empty or ref.Threshold.duplicated().any():raise ValueError('Missing/duplicated DCA reference')
    d=d.merge(ref,on='Threshold',validate='many_to_one');d['Displayed_NB']=(d.Net_Benefit-d.Reference_NB)*1000 if delta else d.Net_Benefit*100
    for rep in REPS:
        q=d[d.Rep.eq(rep)].sort_values('Threshold');ax.plot(q.Threshold*100,q.Displayed_NB,color=COLORS[rep],label=rep)
    if delta:ax.axhline(0,color='.5',ls=':')
    else:
        ax.plot(ref.Threshold*100,ref.Reference_NB*100,color='.25',ls='--',label='Clinical + PRS')
        for label in ['Treat all','Treat none','all','none']:
            q=original[original.Strategy.eq(label)].sort_values('Threshold')
            if not q.empty:ax.plot(q.Threshold*100,q.Net_Benefit*100,ls=':',color='.6',label=label)
    ax.set(xlabel=f'{h}-year risk threshold (%)',ylabel='Incremental net benefit per 1,000' if delta else 'Net benefit per 100',title=f'{h}-year DCA | {alg}');ax.legend(frameon=False,fontsize=9);save(fig,d,stem,'Full available threshold range. Curves are point estimates; confidence intervals unavailable.')

def algorithm_metric(file,e,l,u,title,stem):
    d=four(file,10);fig,axes=plt.subplots(2,2,figsize=(10,7))
    for ax,rep in zip(axes.flat,REPS):
        q=d[d.Rep.eq(rep)].copy();q=q.set_index('Model').reindex([a for a in ALGS if a in q.Model.values]).reset_index()
        v=q[e].to_numpy(float);lo=q[l].to_numpy(float);hi=q[u].to_numpy(float)
        if not np.isfinite(np.r_[v,lo,hi]).all() or np.any(lo>v) or np.any(hi<v):raise ValueError('Invalid metric/CI')
        ax.errorbar(v,np.arange(len(q)),xerr=[v-lo,hi-v],fmt='o',color=COLORS[rep],capsize=3)
        if e.startswith('Delta'):ax.axvline(0,color='.6',ls='--')
        ax.set_yticks(range(len(q)),q.Model);ax.invert_yaxis();ax.set_title(rep);ax.set_xlabel(title);ax.grid(axis='x',alpha=.15)
    save(fig,d,stem,'10-year estimates and 95% CIs. Each panel shows one NMR representation.')

# Subgroup tables are loaded only from their explicitly named run directory.
def sgroot(score):return RUNS[score]/'04_results/reviewer_ethnicity_all_prediction_models'
def sg_table(name):
    parts=[]
    for score in ['Score249','Score106']:
        d=read(sgroot(score)/name);d['Score_Run']=score;parts.append(d)
    return pd.concat(parts,ignore_index=True)
def spec(rep,alg):
    return ('Score106' if rep=='Score106' else 'Score249',FEATURES[rep]+'__'+alg.replace('XGBoost survival','XGBoost Cox'))
def sg_rows(name,h,alg,pair=False,kind=None):
    allrows=sg_table(name);parts=[]
    for rep in REPS:
        score,key=spec(rep,alg);col='New' if pair else 'Model'
        d=allrows[(allrows.Score_Run==score)&(allrows.Horizon==h)&(allrows[col]==key)].copy()
        if kind:d=d[d.Comparison.eq(kind)]
        d['Rep']=rep;parts.append(d)
    d=pd.concat(parts,ignore_index=True)
    if len(d)!=8 or d.duplicated(['Rep','Group']).any():raise ValueError('Expected four representations x two groups')
    return d

def ethnic_forest(h,alg,metric,stem,kind=None):
    fields={'Uno_C':('Lower95','Upper95','Uno C-index'),'Delta_Uno_C':('Delta_Uno_C_Lower95','Delta_Uno_C_Upper95','Change in Uno C-index'),'Delta_IBS':('Delta_IBS_Lower95','Delta_IBS_Upper95','Change in IPCW IBS'),'NRI_Continuous':('NRI_Continuous_Lower95','NRI_Continuous_Upper95','Continuous NRI'),'IDI':('IDI_Lower95','IDI_Upper95','IDI')}
    pair=metric!='Uno_C';d=sg_rows('all_paired_differences.csv' if pair else 'all_model_performance.csv',h,alg,pair,kind)
    lo,hi,label=fields[metric]
    if not np.isfinite(d[[metric,lo,hi]].to_numpy(float)).all():raise ValueError('Metric/CI not computed for this comparison')
    fig,ax=plt.subplots(figsize=(8.5,4.7))
    for gi,g in enumerate(GROUPS):
        q=d[d.Group.eq(g)].set_index('Rep').loc[REPS];v=q[metric].to_numpy();l=q[lo].to_numpy();u=q[hi].to_numpy()
        if np.any(l>v) or np.any(u<v):raise ValueError('Invalid CI')
        ax.errorbar(v,np.arange(4)+(gi-.5)*.22,xerr=[v-l,u-v],fmt=['o','s'][gi],color=GC[g],capsize=3,label=GLABEL[g])
    if pair:ax.axvline(0,color='.5',ls='--',lw=.8)
    ax.set_yticks(range(4),REPS);ax.invert_yaxis();ax.set_xlabel(label+' (95% CI)');ax.set_title(f'{h}-year {label} | {alg}',loc='left');ax.grid(axis='x',alpha=.15);ax.legend(frameon=False,fontsize=9,loc='best')
    save(fig,d,stem,'Exploratory internal subgroups; pointwise CIs. These are not tests of between-group differences.\n'+('Reference: Cox with the same representation.' if kind=='Algorithm_vs_Cox' else 'Reference: Clinical + PRS (Cox PH).' if pair else ''))

def ethnic_points(h,alg,metric,stem):
    d=sg_rows('all_model_performance.csv',h,alg);fig,ax=plt.subplots(figsize=(8,4.5))
    for gi,g in enumerate(GROUPS):
        q=d[d.Group.eq(g)].set_index('Rep').loc[REPS];ax.scatter(q[metric],np.arange(4)+(gi-.5)*.22,color=GC[g],marker=['o','s'][gi],label=GLABEL[g])
    ax.set_yticks(range(4),REPS);ax.invert_yaxis();ax.set_xlabel(metric+' (point estimate)');ax.set_title(f'{h}-year {metric} | {alg}');ax.legend(frameon=False,fontsize=9)
    save(fig,d,stem,'Absolute estimates without CIs in the source. Between-group differences also reflect case mix and incidence.')

def ethnic_cal(h,rep,alg,stem):
    score,key=spec(rep,alg);parts=[]
    for g in GROUPS:
        q=read(sgroot(score)/f'{g}_{h}y_{key}_calibration.csv');q['Group']=g;parts.append(q)
    d=pd.concat(parts);fig,axes=plt.subplots(1,2,figsize=(9,4.5),sharex=True,sharey=True);limit=max(d.Predicted_Risk.max(),d.Observed_KM_Risk.max())*110
    for ax,g in zip(axes,GROUPS):
        q=d[d.Group.eq(g)].sort_values('Predicted_Risk');ax.plot([0,limit],[0,limit],'--',color='.6');ax.plot(q.Predicted_Risk*100,q.Observed_KM_Risk*100,'o-',color=GC[g]);ax.set(title=GLABEL[g],xlabel='Predicted risk (%)',ylabel='Observed KM risk (%)',xlim=(0,limit),ylim=(0,limit))
    fig.suptitle(f'{rep} | {alg} | {h}-year calibration');save(fig,d,stem,'Five predicted-risk groups per subgroup; group boundaries differ. Unsmoothed point estimates; confidence intervals unavailable.')

def ethnic_dca(h,rep,alg,stem,delta=False):
    score,key=spec(rep,alg);parts=[];fig,axes=plt.subplots(1,2,figsize=(10,4.5),sharex=True,sharey=True)
    for ax,g in zip(axes,GROUPS):
        d=read(sgroot(score)/f'{g}_{h}y_dca.csv');q=d[d.Strategy.eq(key)].copy().sort_values('Threshold');ref=d[d.Strategy.eq('Clinical_plus_PRS__Cox PH')][['Threshold','Net_Benefit']].rename(columns={'Net_Benefit':'Reference_NB'})
        if q.empty:raise ValueError('Missing DCA strategy '+key)
        q=q.merge(ref,on='Threshold',validate='one_to_one');q['Group']=g;q['Shown']=1000*(q.Net_Benefit-q.Reference_NB) if delta else 100*q.Net_Benefit;parts.append(q)
        ax.plot(q.Threshold*100,q.Shown,color=GC[g],lw=1.7,label=rep+' / '+alg)
        if delta:ax.axhline(0,ls=':',color='.5')
        else:
            ax.plot(q.Threshold*100,q.Reference_NB*100,ls='--',color='.3',label='Clinical + PRS')
            for tag in ['all','none','Treat all','Treat none']:
                r=d[d.Strategy.eq(tag)].sort_values('Threshold')
                if not r.empty:ax.plot(r.Threshold*100,r.Net_Benefit*100,ls=':',color='.6',label=tag)
        ax.set(title=GLABEL[g],xlabel=f'{h}-year risk threshold (%)',ylabel='Incremental NB per 1,000' if delta else 'Net benefit per 100');ax.legend(frameon=False,fontsize=7)
    save(fig,pd.concat(parts),stem,'Full available threshold range; point estimates only. Incremental reference: Clinical + PRS Cox.\nAbsolute net benefit differs with event incidence; curves do not establish between-group superiority.')
print('Output:',OUT)

def dimensions(stem):
    d=four('figure3A_representation_scale.csv').rename(columns={'NMR_Predictor_N':'Inputs'})
    d=d.set_index('Rep').loc[REPS].reset_index()
    fig,ax=plt.subplots(figsize=(7,4))
    bars=ax.bar(d.Rep,d.Inputs,color=[COLORS[r] for r in REPS],width=.55)
    for b,v in zip(bars,d.Inputs):ax.text(b.get_x()+b.get_width()/2,v+2,str(v),ha='center')
    ax.set_ylim(0,120);ax.set_ylabel('NMR inputs in the downstream model');ax.set_title('NMR representation dimensionality',loc='left')
    save(fig,d,stem,'Scores are each one input; candidate panels contain 249 and 106 variables, respectively.\nDimensions are read from the saved representation-scale tables.')

def ethnic_core(h,score,metric,stem):
    d=read(sgroot(score)/'all_model_performance.csv');keys=['Clinical_only__Cox PH','Clinical_plus_PRS__Cox PH','Clinical_plus_NMR_score__Cox PH','Clinical_plus_PRS_plus_NMR_score__Cox PH']
    d=d[(d.Horizon==h)&d.Model.isin(keys)].copy();labels=['Clinical','Clinical + PRS','Clinical + '+score,'Clinical + PRS + '+score]
    fig,ax=plt.subplots(figsize=(8,4.7))
    for gi,g in enumerate(GROUPS):
        q=d[d.Group.eq(g)].set_index('Model').loc[keys];v=q[metric].to_numpy();y=np.arange(4)+(gi-.5)*.22
        if metric=='Uno_C':ax.errorbar(v,y,xerr=[v-q.Lower95.to_numpy(),q.Upper95.to_numpy()-v],fmt=['o','s'][gi],color=GC[g],capsize=3,label=GLABEL[g])
        else:ax.scatter(v,y,marker=['o','s'][gi],color=GC[g],label=GLABEL[g])
    ax.set_yticks(range(4),labels);ax.invert_yaxis();ax.set_xlabel('Uno C-index (95% CI)' if metric=='Uno_C' else 'IPCW IBS (point estimate)');ax.set_title(f'{h}-year core Cox models | {score}');ax.legend(frameon=False,fontsize=9)
    save(fig,d,stem,'Exploratory internal ethnic subgroups; intervals do not test between-group differences.')


In [ ]:
run("figure3A_four_dimensions",dimensions)


In [ ]:
# figure3B_four_5y_Cox_PH
run('figure3B_four_5y_Cox_PH',global_metric,'figure3B_delta_uno_cindex_vs_clinical_prs.csv',5,'Cox PH','Delta_Uno_C','Delta_Uno_C_Lower95','Delta_Uno_C_Upper95','Delta Uno C-index')


In [ ]:
# figure3D_four_5y_Cox_PH
run('figure3D_four_5y_Cox_PH',global_metric,'figure3D_delta_ipcw_ibs.csv',5,'Cox PH','Delta_IBS','Delta_IBS_Lower95','Delta_IBS_Upper95','Delta IPCW IBS')


In [ ]:
# figure3E_NRI_four_5y_Cox_PH
run('figure3E_NRI_four_5y_Cox_PH',global_metric,'figure3E_ipcw_nri_idi.csv',5,'Cox PH','NRI_Continuous','NRI_Continuous_Lower95','NRI_Continuous_Upper95','Continuous NRI')


In [ ]:
# figure3E_IDI_four_5y_Cox_PH
run('figure3E_IDI_four_5y_Cox_PH',global_metric,'figure3E_ipcw_nri_idi.csv',5,'Cox PH','IDI','IDI_Lower95','IDI_Upper95','IDI')


In [ ]:
# figure3F_calibration_four_5y_Cox_PH
run('figure3F_calibration_four_5y_Cox_PH',global_cal,5,'Cox PH')


In [ ]:
# figure3G_DCA_four_5y_Cox_PH
run('figure3G_DCA_four_5y_Cox_PH',global_dca,5,'Cox PH')


In [ ]:
# figure3G_deltaNB_four_5y_Cox_PH
run('figure3G_deltaNB_four_5y_Cox_PH',global_dca,5,'Cox PH',delta=True)


In [ ]:
# figure3B_four_5y_Elastic-net_Cox
run('figure3B_four_5y_Elastic-net_Cox',global_metric,'figure3B_delta_uno_cindex_vs_clinical_prs.csv',5,'Elastic-net Cox','Delta_Uno_C','Delta_Uno_C_Lower95','Delta_Uno_C_Upper95','Delta Uno C-index')


In [ ]:
# figure3D_four_5y_Elastic-net_Cox
run('figure3D_four_5y_Elastic-net_Cox',global_metric,'figure3D_delta_ipcw_ibs.csv',5,'Elastic-net Cox','Delta_IBS','Delta_IBS_Lower95','Delta_IBS_Upper95','Delta IPCW IBS')


In [ ]:
# figure3E_NRI_four_5y_Elastic-net_Cox
run('figure3E_NRI_four_5y_Elastic-net_Cox',global_metric,'figure3E_ipcw_nri_idi.csv',5,'Elastic-net Cox','NRI_Continuous','NRI_Continuous_Lower95','NRI_Continuous_Upper95','Continuous NRI')


In [ ]:
# figure3E_IDI_four_5y_Elastic-net_Cox
run('figure3E_IDI_four_5y_Elastic-net_Cox',global_metric,'figure3E_ipcw_nri_idi.csv',5,'Elastic-net Cox','IDI','IDI_Lower95','IDI_Upper95','IDI')


In [ ]:
# figure3F_calibration_four_5y_Elastic-net_Cox
run('figure3F_calibration_four_5y_Elastic-net_Cox',global_cal,5,'Elastic-net Cox')


In [ ]:
# figure3G_DCA_four_5y_Elastic-net_Cox
run('figure3G_DCA_four_5y_Elastic-net_Cox',global_dca,5,'Elastic-net Cox')


In [ ]:
# figure3G_deltaNB_four_5y_Elastic-net_Cox
run('figure3G_deltaNB_four_5y_Elastic-net_Cox',global_dca,5,'Elastic-net Cox',delta=True)


In [ ]:
# figure3C_four_5y
run('figure3C_four_5y',global_metric,"figure3C_elasticnet_minus_cox_delta_uno_cindex.csv",5,None,'Delta_Uno_C','Delta_Uno_C_Lower95','Delta_Uno_C_Upper95',"Elastic-net minus Cox: delta C")


In [ ]:
# figure3B_four_10y_Cox_PH
run('figure3B_four_10y_Cox_PH',global_metric,'figure3B_delta_uno_cindex_vs_clinical_prs.csv',10,'Cox PH','Delta_Uno_C','Delta_Uno_C_Lower95','Delta_Uno_C_Upper95','Delta Uno C-index')


In [ ]:
# figure3D_four_10y_Cox_PH
run('figure3D_four_10y_Cox_PH',global_metric,'figure3D_delta_ipcw_ibs.csv',10,'Cox PH','Delta_IBS','Delta_IBS_Lower95','Delta_IBS_Upper95','Delta IPCW IBS')


In [ ]:
# figure3E_NRI_four_10y_Cox_PH
run('figure3E_NRI_four_10y_Cox_PH',global_metric,'figure3E_ipcw_nri_idi.csv',10,'Cox PH','NRI_Continuous','NRI_Continuous_Lower95','NRI_Continuous_Upper95','Continuous NRI')


In [ ]:
# figure3E_IDI_four_10y_Cox_PH
run('figure3E_IDI_four_10y_Cox_PH',global_metric,'figure3E_ipcw_nri_idi.csv',10,'Cox PH','IDI','IDI_Lower95','IDI_Upper95','IDI')


In [ ]:
# figure3F_calibration_four_10y_Cox_PH
run('figure3F_calibration_four_10y_Cox_PH',global_cal,10,'Cox PH')


In [ ]:
# figure3G_DCA_four_10y_Cox_PH
run('figure3G_DCA_four_10y_Cox_PH',global_dca,10,'Cox PH')


In [ ]:
# figure3G_deltaNB_four_10y_Cox_PH
run('figure3G_deltaNB_four_10y_Cox_PH',global_dca,10,'Cox PH',delta=True)


In [ ]:
# figure3B_four_10y_Elastic-net_Cox
run('figure3B_four_10y_Elastic-net_Cox',global_metric,'figure3B_delta_uno_cindex_vs_clinical_prs.csv',10,'Elastic-net Cox','Delta_Uno_C','Delta_Uno_C_Lower95','Delta_Uno_C_Upper95','Delta Uno C-index')


In [ ]:
# figure3D_four_10y_Elastic-net_Cox
run('figure3D_four_10y_Elastic-net_Cox',global_metric,'figure3D_delta_ipcw_ibs.csv',10,'Elastic-net Cox','Delta_IBS','Delta_IBS_Lower95','Delta_IBS_Upper95','Delta IPCW IBS')


In [ ]:
# figure3E_NRI_four_10y_Elastic-net_Cox
run('figure3E_NRI_four_10y_Elastic-net_Cox',global_metric,'figure3E_ipcw_nri_idi.csv',10,'Elastic-net Cox','NRI_Continuous','NRI_Continuous_Lower95','NRI_Continuous_Upper95','Continuous NRI')


In [ ]:
# figure3E_IDI_four_10y_Elastic-net_Cox
run('figure3E_IDI_four_10y_Elastic-net_Cox',global_metric,'figure3E_ipcw_nri_idi.csv',10,'Elastic-net Cox','IDI','IDI_Lower95','IDI_Upper95','IDI')


In [ ]:
# figure3F_calibration_four_10y_Elastic-net_Cox
run('figure3F_calibration_four_10y_Elastic-net_Cox',global_cal,10,'Elastic-net Cox')


In [ ]:
# figure3G_DCA_four_10y_Elastic-net_Cox
run('figure3G_DCA_four_10y_Elastic-net_Cox',global_dca,10,'Elastic-net Cox')


In [ ]:
# figure3G_deltaNB_four_10y_Elastic-net_Cox
run('figure3G_deltaNB_four_10y_Elastic-net_Cox',global_dca,10,'Elastic-net Cox',delta=True)


In [ ]:
# figure3C_four_10y
run('figure3C_four_10y',global_metric,"figure3C_elasticnet_minus_cox_delta_uno_cindex.csv",10,None,'Delta_Uno_C','Delta_Uno_C_Lower95','Delta_Uno_C_Upper95',"Elastic-net minus Cox: delta C")


In [ ]:
# figure4A_four_absolute_C
run('figure4A_four_absolute_C',algorithm_metric,'result4_all_models_uno_ipcw_summary.csv','Uno_C','Uno_C_Lower95','Uno_C_Upper95','Uno C-index')


In [ ]:
# figure4B_four_delta_C
run('figure4B_four_delta_C',algorithm_metric,'result4_all_models_delta_uno_ipcw_vs_cox.csv','Delta_Uno_C_vs_Cox','Delta_Uno_C_Lower95','Delta_Uno_C_Upper95','Delta C vs Cox')


In [ ]:
# supp_four_absolute_IBS
run('supp_four_absolute_IBS',algorithm_metric,'result4_all_models_10y_ipcw_ibs.csv','IPCW_IBS','IPCW_IBS_Lower95','IPCW_IBS_Upper95','IPCW IBS')


In [ ]:
# supp_four_delta_IBS
run('supp_four_delta_IBS',algorithm_metric,'result4_all_models_delta_10y_ipcw_ibs_vs_cox.csv','Delta_IPCW_IBS_vs_Cox','Delta_IPCW_IBS_Lower95','Delta_IPCW_IBS_Upper95','Delta IBS vs Cox')


In [ ]:
# ethnicity_5y_Cox_PH_C
run('ethnicity_5y_Cox_PH_C',ethnic_forest,5,'Cox PH',"Uno_C")


In [ ]:
# ethnicity_5y_Cox_PH_IBS
run('ethnicity_5y_Cox_PH_IBS',ethnic_points,5,'Cox PH',"IBS")


In [ ]:
# ethnicity_5y_Elastic-net_Cox_C
run('ethnicity_5y_Elastic-net_Cox_C',ethnic_forest,5,'Elastic-net Cox',"Uno_C")


In [ ]:
# ethnicity_5y_Elastic-net_Cox_IBS
run('ethnicity_5y_Elastic-net_Cox_IBS',ethnic_points,5,'Elastic-net Cox',"IBS")


In [ ]:
# ethnicity_5y_Elastic-net_Cox_Delta_Uno_C_vsCox
run('ethnicity_5y_Elastic-net_Cox_Delta_Uno_C_vsCox',ethnic_forest,5,'Elastic-net Cox','Delta_Uno_C',kind="Algorithm_vs_Cox")


In [ ]:
# ethnicity_5y_Elastic-net_Cox_Delta_IBS_vsCox
run('ethnicity_5y_Elastic-net_Cox_Delta_IBS_vsCox',ethnic_forest,5,'Elastic-net Cox','Delta_IBS',kind="Algorithm_vs_Cox")


In [ ]:
# ethnicity_5y_XGBoost_survival_C
run('ethnicity_5y_XGBoost_survival_C',ethnic_forest,5,'XGBoost survival',"Uno_C")


In [ ]:
# ethnicity_5y_XGBoost_survival_IBS
run('ethnicity_5y_XGBoost_survival_IBS',ethnic_points,5,'XGBoost survival',"IBS")


In [ ]:
# ethnicity_5y_XGBoost_survival_Delta_Uno_C_vsCox
run('ethnicity_5y_XGBoost_survival_Delta_Uno_C_vsCox',ethnic_forest,5,'XGBoost survival','Delta_Uno_C',kind="Algorithm_vs_Cox")


In [ ]:
# ethnicity_5y_XGBoost_survival_Delta_IBS_vsCox
run('ethnicity_5y_XGBoost_survival_Delta_IBS_vsCox',ethnic_forest,5,'XGBoost survival','Delta_IBS',kind="Algorithm_vs_Cox")


In [ ]:
# ethnicity_5y_DeepSurv_C
run('ethnicity_5y_DeepSurv_C',ethnic_forest,5,'DeepSurv',"Uno_C")


In [ ]:
# ethnicity_5y_DeepSurv_IBS
run('ethnicity_5y_DeepSurv_IBS',ethnic_points,5,'DeepSurv',"IBS")


In [ ]:
# ethnicity_5y_DeepSurv_Delta_Uno_C_vsCox
run('ethnicity_5y_DeepSurv_Delta_Uno_C_vsCox',ethnic_forest,5,'DeepSurv','Delta_Uno_C',kind="Algorithm_vs_Cox")


In [ ]:
# ethnicity_5y_DeepSurv_Delta_IBS_vsCox
run('ethnicity_5y_DeepSurv_Delta_IBS_vsCox',ethnic_forest,5,'DeepSurv','Delta_IBS',kind="Algorithm_vs_Cox")


In [ ]:
# ethnicity_5y_Cox_Delta_Uno_C_vsClinicalPRS
run('ethnicity_5y_Cox_Delta_Uno_C_vsClinicalPRS',ethnic_forest,5,"Cox PH",'Delta_Uno_C',kind="Representation_vs_Clinical_PRS")


In [ ]:
# ethnicity_5y_Cox_Delta_IBS_vsClinicalPRS
run('ethnicity_5y_Cox_Delta_IBS_vsClinicalPRS',ethnic_forest,5,"Cox PH",'Delta_IBS',kind="Representation_vs_Clinical_PRS")


In [ ]:
# ethnicity_5y_Cox_NRI_Continuous_vsClinicalPRS
run('ethnicity_5y_Cox_NRI_Continuous_vsClinicalPRS',ethnic_forest,5,"Cox PH",'NRI_Continuous',kind="Representation_vs_Clinical_PRS")


In [ ]:
# ethnicity_5y_Cox_IDI_vsClinicalPRS
run('ethnicity_5y_Cox_IDI_vsClinicalPRS',ethnic_forest,5,"Cox PH",'IDI',kind="Representation_vs_Clinical_PRS")


In [ ]:
# ethnicity_10y_Cox_PH_C
run('ethnicity_10y_Cox_PH_C',ethnic_forest,10,'Cox PH',"Uno_C")


In [ ]:
# ethnicity_10y_Cox_PH_IBS
run('ethnicity_10y_Cox_PH_IBS',ethnic_points,10,'Cox PH',"IBS")


In [ ]:
# ethnicity_10y_Elastic-net_Cox_C
run('ethnicity_10y_Elastic-net_Cox_C',ethnic_forest,10,'Elastic-net Cox',"Uno_C")


In [ ]:
# ethnicity_10y_Elastic-net_Cox_IBS
run('ethnicity_10y_Elastic-net_Cox_IBS',ethnic_points,10,'Elastic-net Cox',"IBS")


In [ ]:
# ethnicity_10y_Elastic-net_Cox_Delta_Uno_C_vsCox
run('ethnicity_10y_Elastic-net_Cox_Delta_Uno_C_vsCox',ethnic_forest,10,'Elastic-net Cox','Delta_Uno_C',kind="Algorithm_vs_Cox")


In [ ]:
# ethnicity_10y_Elastic-net_Cox_Delta_IBS_vsCox
run('ethnicity_10y_Elastic-net_Cox_Delta_IBS_vsCox',ethnic_forest,10,'Elastic-net Cox','Delta_IBS',kind="Algorithm_vs_Cox")


In [ ]:
# ethnicity_10y_XGBoost_survival_C
run('ethnicity_10y_XGBoost_survival_C',ethnic_forest,10,'XGBoost survival',"Uno_C")


In [ ]:
# ethnicity_10y_XGBoost_survival_IBS
run('ethnicity_10y_XGBoost_survival_IBS',ethnic_points,10,'XGBoost survival',"IBS")


In [ ]:
# ethnicity_10y_XGBoost_survival_Delta_Uno_C_vsCox
run('ethnicity_10y_XGBoost_survival_Delta_Uno_C_vsCox',ethnic_forest,10,'XGBoost survival','Delta_Uno_C',kind="Algorithm_vs_Cox")


In [ ]:
# ethnicity_10y_XGBoost_survival_Delta_IBS_vsCox
run('ethnicity_10y_XGBoost_survival_Delta_IBS_vsCox',ethnic_forest,10,'XGBoost survival','Delta_IBS',kind="Algorithm_vs_Cox")


In [ ]:
# ethnicity_10y_DeepSurv_C
run('ethnicity_10y_DeepSurv_C',ethnic_forest,10,'DeepSurv',"Uno_C")


In [ ]:
# ethnicity_10y_DeepSurv_IBS
run('ethnicity_10y_DeepSurv_IBS',ethnic_points,10,'DeepSurv',"IBS")


In [ ]:
# ethnicity_10y_DeepSurv_Delta_Uno_C_vsCox
run('ethnicity_10y_DeepSurv_Delta_Uno_C_vsCox',ethnic_forest,10,'DeepSurv','Delta_Uno_C',kind="Algorithm_vs_Cox")


In [ ]:
# ethnicity_10y_DeepSurv_Delta_IBS_vsCox
run('ethnicity_10y_DeepSurv_Delta_IBS_vsCox',ethnic_forest,10,'DeepSurv','Delta_IBS',kind="Algorithm_vs_Cox")


In [ ]:
# ethnicity_10y_Cox_Delta_Uno_C_vsClinicalPRS
run('ethnicity_10y_Cox_Delta_Uno_C_vsClinicalPRS',ethnic_forest,10,"Cox PH",'Delta_Uno_C',kind="Representation_vs_Clinical_PRS")


In [ ]:
# ethnicity_10y_Cox_Delta_IBS_vsClinicalPRS
run('ethnicity_10y_Cox_Delta_IBS_vsClinicalPRS',ethnic_forest,10,"Cox PH",'Delta_IBS',kind="Representation_vs_Clinical_PRS")


In [ ]:
# ethnicity_10y_Cox_NRI_Continuous_vsClinicalPRS
run('ethnicity_10y_Cox_NRI_Continuous_vsClinicalPRS',ethnic_forest,10,"Cox PH",'NRI_Continuous',kind="Representation_vs_Clinical_PRS")


In [ ]:
# ethnicity_10y_Cox_IDI_vsClinicalPRS
run('ethnicity_10y_Cox_IDI_vsClinicalPRS',ethnic_forest,10,"Cox PH",'IDI',kind="Representation_vs_Clinical_PRS")


In [ ]:
# ethnicity_5y_Score249_Cox_PH_calibration
run('ethnicity_5y_Score249_Cox_PH_calibration',ethnic_cal,5,'Score249','Cox PH')


In [ ]:
# ethnicity_5y_Score249_Cox_PH_DCA
run('ethnicity_5y_Score249_Cox_PH_DCA',ethnic_dca,5,'Score249','Cox PH')


In [ ]:
# ethnicity_5y_Score249_Cox_PH_deltaNB
run('ethnicity_5y_Score249_Cox_PH_deltaNB',ethnic_dca,5,'Score249','Cox PH',delta=True)


In [ ]:
# ethnicity_5y_Score249_Elastic-net_Cox_calibration
run('ethnicity_5y_Score249_Elastic-net_Cox_calibration',ethnic_cal,5,'Score249','Elastic-net Cox')


In [ ]:
# ethnicity_5y_Score249_Elastic-net_Cox_DCA
run('ethnicity_5y_Score249_Elastic-net_Cox_DCA',ethnic_dca,5,'Score249','Elastic-net Cox')


In [ ]:
# ethnicity_5y_Score249_Elastic-net_Cox_deltaNB
run('ethnicity_5y_Score249_Elastic-net_Cox_deltaNB',ethnic_dca,5,'Score249','Elastic-net Cox',delta=True)


In [ ]:
# ethnicity_5y_Score249_XGBoost_survival_calibration
run('ethnicity_5y_Score249_XGBoost_survival_calibration',ethnic_cal,5,'Score249','XGBoost survival')


In [ ]:
# ethnicity_5y_Score249_XGBoost_survival_DCA
run('ethnicity_5y_Score249_XGBoost_survival_DCA',ethnic_dca,5,'Score249','XGBoost survival')


In [ ]:
# ethnicity_5y_Score249_XGBoost_survival_deltaNB
run('ethnicity_5y_Score249_XGBoost_survival_deltaNB',ethnic_dca,5,'Score249','XGBoost survival',delta=True)


In [ ]:
# ethnicity_5y_Score249_DeepSurv_calibration
run('ethnicity_5y_Score249_DeepSurv_calibration',ethnic_cal,5,'Score249','DeepSurv')


In [ ]:
# ethnicity_5y_Score249_DeepSurv_DCA
run('ethnicity_5y_Score249_DeepSurv_DCA',ethnic_dca,5,'Score249','DeepSurv')


In [ ]:
# ethnicity_5y_Score249_DeepSurv_deltaNB
run('ethnicity_5y_Score249_DeepSurv_deltaNB',ethnic_dca,5,'Score249','DeepSurv',delta=True)


In [ ]:
# ethnicity_5y_Score106_Cox_PH_calibration
run('ethnicity_5y_Score106_Cox_PH_calibration',ethnic_cal,5,'Score106','Cox PH')


In [ ]:
# ethnicity_5y_Score106_Cox_PH_DCA
run('ethnicity_5y_Score106_Cox_PH_DCA',ethnic_dca,5,'Score106','Cox PH')


In [ ]:
# ethnicity_5y_Score106_Cox_PH_deltaNB
run('ethnicity_5y_Score106_Cox_PH_deltaNB',ethnic_dca,5,'Score106','Cox PH',delta=True)


In [ ]:
# ethnicity_5y_Score106_Elastic-net_Cox_calibration
run('ethnicity_5y_Score106_Elastic-net_Cox_calibration',ethnic_cal,5,'Score106','Elastic-net Cox')


In [ ]:
# ethnicity_5y_Score106_Elastic-net_Cox_DCA
run('ethnicity_5y_Score106_Elastic-net_Cox_DCA',ethnic_dca,5,'Score106','Elastic-net Cox')


In [ ]:
# ethnicity_5y_Score106_Elastic-net_Cox_deltaNB
run('ethnicity_5y_Score106_Elastic-net_Cox_deltaNB',ethnic_dca,5,'Score106','Elastic-net Cox',delta=True)


In [ ]:
# ethnicity_5y_Score106_XGBoost_survival_calibration
run('ethnicity_5y_Score106_XGBoost_survival_calibration',ethnic_cal,5,'Score106','XGBoost survival')


In [ ]:
# ethnicity_5y_Score106_XGBoost_survival_DCA
run('ethnicity_5y_Score106_XGBoost_survival_DCA',ethnic_dca,5,'Score106','XGBoost survival')


In [ ]:
# ethnicity_5y_Score106_XGBoost_survival_deltaNB
run('ethnicity_5y_Score106_XGBoost_survival_deltaNB',ethnic_dca,5,'Score106','XGBoost survival',delta=True)


In [ ]:
# ethnicity_5y_Score106_DeepSurv_calibration
run('ethnicity_5y_Score106_DeepSurv_calibration',ethnic_cal,5,'Score106','DeepSurv')


In [ ]:
# ethnicity_5y_Score106_DeepSurv_DCA
run('ethnicity_5y_Score106_DeepSurv_DCA',ethnic_dca,5,'Score106','DeepSurv')


In [ ]:
# ethnicity_5y_Score106_DeepSurv_deltaNB
run('ethnicity_5y_Score106_DeepSurv_deltaNB',ethnic_dca,5,'Score106','DeepSurv',delta=True)


In [ ]:
# ethnicity_5y_Spearman-pruned_Cox_PH_calibration
run('ethnicity_5y_Spearman-pruned_Cox_PH_calibration',ethnic_cal,5,'Spearman-pruned','Cox PH')


In [ ]:
# ethnicity_5y_Spearman-pruned_Cox_PH_DCA
run('ethnicity_5y_Spearman-pruned_Cox_PH_DCA',ethnic_dca,5,'Spearman-pruned','Cox PH')


In [ ]:
# ethnicity_5y_Spearman-pruned_Cox_PH_deltaNB
run('ethnicity_5y_Spearman-pruned_Cox_PH_deltaNB',ethnic_dca,5,'Spearman-pruned','Cox PH',delta=True)


In [ ]:
# ethnicity_5y_Spearman-pruned_Elastic-net_Cox_calibration
run('ethnicity_5y_Spearman-pruned_Elastic-net_Cox_calibration',ethnic_cal,5,'Spearman-pruned','Elastic-net Cox')


In [ ]:
# ethnicity_5y_Spearman-pruned_Elastic-net_Cox_DCA
run('ethnicity_5y_Spearman-pruned_Elastic-net_Cox_DCA',ethnic_dca,5,'Spearman-pruned','Elastic-net Cox')


In [ ]:
# ethnicity_5y_Spearman-pruned_Elastic-net_Cox_deltaNB
run('ethnicity_5y_Spearman-pruned_Elastic-net_Cox_deltaNB',ethnic_dca,5,'Spearman-pruned','Elastic-net Cox',delta=True)


In [ ]:
# ethnicity_5y_Spearman-pruned_XGBoost_survival_calibration
run('ethnicity_5y_Spearman-pruned_XGBoost_survival_calibration',ethnic_cal,5,'Spearman-pruned','XGBoost survival')


In [ ]:
# ethnicity_5y_Spearman-pruned_XGBoost_survival_DCA
run('ethnicity_5y_Spearman-pruned_XGBoost_survival_DCA',ethnic_dca,5,'Spearman-pruned','XGBoost survival')


In [ ]:
# ethnicity_5y_Spearman-pruned_XGBoost_survival_deltaNB
run('ethnicity_5y_Spearman-pruned_XGBoost_survival_deltaNB',ethnic_dca,5,'Spearman-pruned','XGBoost survival',delta=True)


In [ ]:
# ethnicity_5y_Spearman-pruned_DeepSurv_calibration
run('ethnicity_5y_Spearman-pruned_DeepSurv_calibration',ethnic_cal,5,'Spearman-pruned','DeepSurv')


In [ ]:
# ethnicity_5y_Spearman-pruned_DeepSurv_DCA
run('ethnicity_5y_Spearman-pruned_DeepSurv_DCA',ethnic_dca,5,'Spearman-pruned','DeepSurv')


In [ ]:
# ethnicity_5y_Spearman-pruned_DeepSurv_deltaNB
run('ethnicity_5y_Spearman-pruned_DeepSurv_deltaNB',ethnic_dca,5,'Spearman-pruned','DeepSurv',delta=True)


In [ ]:
# ethnicity_5y_Raw106_Cox_PH_calibration
run('ethnicity_5y_Raw106_Cox_PH_calibration',ethnic_cal,5,'Raw106','Cox PH')


In [ ]:
# ethnicity_5y_Raw106_Cox_PH_DCA
run('ethnicity_5y_Raw106_Cox_PH_DCA',ethnic_dca,5,'Raw106','Cox PH')


In [ ]:
# ethnicity_5y_Raw106_Cox_PH_deltaNB
run('ethnicity_5y_Raw106_Cox_PH_deltaNB',ethnic_dca,5,'Raw106','Cox PH',delta=True)


In [ ]:
# ethnicity_5y_Raw106_Elastic-net_Cox_calibration
run('ethnicity_5y_Raw106_Elastic-net_Cox_calibration',ethnic_cal,5,'Raw106','Elastic-net Cox')


In [ ]:
# ethnicity_5y_Raw106_Elastic-net_Cox_DCA
run('ethnicity_5y_Raw106_Elastic-net_Cox_DCA',ethnic_dca,5,'Raw106','Elastic-net Cox')


In [ ]:
# ethnicity_5y_Raw106_Elastic-net_Cox_deltaNB
run('ethnicity_5y_Raw106_Elastic-net_Cox_deltaNB',ethnic_dca,5,'Raw106','Elastic-net Cox',delta=True)


In [ ]:
# ethnicity_5y_Raw106_XGBoost_survival_calibration
run('ethnicity_5y_Raw106_XGBoost_survival_calibration',ethnic_cal,5,'Raw106','XGBoost survival')


In [ ]:
# ethnicity_5y_Raw106_XGBoost_survival_DCA
run('ethnicity_5y_Raw106_XGBoost_survival_DCA',ethnic_dca,5,'Raw106','XGBoost survival')


In [ ]:
# ethnicity_5y_Raw106_XGBoost_survival_deltaNB
run('ethnicity_5y_Raw106_XGBoost_survival_deltaNB',ethnic_dca,5,'Raw106','XGBoost survival',delta=True)


In [ ]:
# ethnicity_5y_Raw106_DeepSurv_calibration
run('ethnicity_5y_Raw106_DeepSurv_calibration',ethnic_cal,5,'Raw106','DeepSurv')


In [ ]:
# ethnicity_5y_Raw106_DeepSurv_DCA
run('ethnicity_5y_Raw106_DeepSurv_DCA',ethnic_dca,5,'Raw106','DeepSurv')


In [ ]:
# ethnicity_5y_Raw106_DeepSurv_deltaNB
run('ethnicity_5y_Raw106_DeepSurv_deltaNB',ethnic_dca,5,'Raw106','DeepSurv',delta=True)


In [ ]:
# ethnicity_10y_Score249_Cox_PH_calibration
run('ethnicity_10y_Score249_Cox_PH_calibration',ethnic_cal,10,'Score249','Cox PH')


In [ ]:
# ethnicity_10y_Score249_Cox_PH_DCA
run('ethnicity_10y_Score249_Cox_PH_DCA',ethnic_dca,10,'Score249','Cox PH')


In [ ]:
# ethnicity_10y_Score249_Cox_PH_deltaNB
run('ethnicity_10y_Score249_Cox_PH_deltaNB',ethnic_dca,10,'Score249','Cox PH',delta=True)


In [ ]:
# ethnicity_10y_Score249_Elastic-net_Cox_calibration
run('ethnicity_10y_Score249_Elastic-net_Cox_calibration',ethnic_cal,10,'Score249','Elastic-net Cox')


In [ ]:
# ethnicity_10y_Score249_Elastic-net_Cox_DCA
run('ethnicity_10y_Score249_Elastic-net_Cox_DCA',ethnic_dca,10,'Score249','Elastic-net Cox')


In [ ]:
# ethnicity_10y_Score249_Elastic-net_Cox_deltaNB
run('ethnicity_10y_Score249_Elastic-net_Cox_deltaNB',ethnic_dca,10,'Score249','Elastic-net Cox',delta=True)


In [ ]:
# ethnicity_10y_Score249_XGBoost_survival_calibration
run('ethnicity_10y_Score249_XGBoost_survival_calibration',ethnic_cal,10,'Score249','XGBoost survival')


In [ ]:
# ethnicity_10y_Score249_XGBoost_survival_DCA
run('ethnicity_10y_Score249_XGBoost_survival_DCA',ethnic_dca,10,'Score249','XGBoost survival')


In [ ]:
# ethnicity_10y_Score249_XGBoost_survival_deltaNB
run('ethnicity_10y_Score249_XGBoost_survival_deltaNB',ethnic_dca,10,'Score249','XGBoost survival',delta=True)


In [ ]:
# ethnicity_10y_Score249_DeepSurv_calibration
run('ethnicity_10y_Score249_DeepSurv_calibration',ethnic_cal,10,'Score249','DeepSurv')


In [ ]:
# ethnicity_10y_Score249_DeepSurv_DCA
run('ethnicity_10y_Score249_DeepSurv_DCA',ethnic_dca,10,'Score249','DeepSurv')


In [ ]:
# ethnicity_10y_Score249_DeepSurv_deltaNB
run('ethnicity_10y_Score249_DeepSurv_deltaNB',ethnic_dca,10,'Score249','DeepSurv',delta=True)


In [ ]:
# ethnicity_10y_Score106_Cox_PH_calibration
run('ethnicity_10y_Score106_Cox_PH_calibration',ethnic_cal,10,'Score106','Cox PH')


In [ ]:
# ethnicity_10y_Score106_Cox_PH_DCA
run('ethnicity_10y_Score106_Cox_PH_DCA',ethnic_dca,10,'Score106','Cox PH')


In [ ]:
# ethnicity_10y_Score106_Cox_PH_deltaNB
run('ethnicity_10y_Score106_Cox_PH_deltaNB',ethnic_dca,10,'Score106','Cox PH',delta=True)


In [ ]:
# ethnicity_10y_Score106_Elastic-net_Cox_calibration
run('ethnicity_10y_Score106_Elastic-net_Cox_calibration',ethnic_cal,10,'Score106','Elastic-net Cox')


In [ ]:
# ethnicity_10y_Score106_Elastic-net_Cox_DCA
run('ethnicity_10y_Score106_Elastic-net_Cox_DCA',ethnic_dca,10,'Score106','Elastic-net Cox')


In [ ]:
# ethnicity_10y_Score106_Elastic-net_Cox_deltaNB
run('ethnicity_10y_Score106_Elastic-net_Cox_deltaNB',ethnic_dca,10,'Score106','Elastic-net Cox',delta=True)


In [ ]:
# ethnicity_10y_Score106_XGBoost_survival_calibration
run('ethnicity_10y_Score106_XGBoost_survival_calibration',ethnic_cal,10,'Score106','XGBoost survival')


In [ ]:
# ethnicity_10y_Score106_XGBoost_survival_DCA
run('ethnicity_10y_Score106_XGBoost_survival_DCA',ethnic_dca,10,'Score106','XGBoost survival')


In [ ]:
# ethnicity_10y_Score106_XGBoost_survival_deltaNB
run('ethnicity_10y_Score106_XGBoost_survival_deltaNB',ethnic_dca,10,'Score106','XGBoost survival',delta=True)


In [ ]:
# ethnicity_10y_Score106_DeepSurv_calibration
run('ethnicity_10y_Score106_DeepSurv_calibration',ethnic_cal,10,'Score106','DeepSurv')


In [ ]:
# ethnicity_10y_Score106_DeepSurv_DCA
run('ethnicity_10y_Score106_DeepSurv_DCA',ethnic_dca,10,'Score106','DeepSurv')


In [ ]:
# ethnicity_10y_Score106_DeepSurv_deltaNB
run('ethnicity_10y_Score106_DeepSurv_deltaNB',ethnic_dca,10,'Score106','DeepSurv',delta=True)


In [ ]:
# ethnicity_10y_Spearman-pruned_Cox_PH_calibration
run('ethnicity_10y_Spearman-pruned_Cox_PH_calibration',ethnic_cal,10,'Spearman-pruned','Cox PH')


In [ ]:
# ethnicity_10y_Spearman-pruned_Cox_PH_DCA
run('ethnicity_10y_Spearman-pruned_Cox_PH_DCA',ethnic_dca,10,'Spearman-pruned','Cox PH')


In [ ]:
# ethnicity_10y_Spearman-pruned_Cox_PH_deltaNB
run('ethnicity_10y_Spearman-pruned_Cox_PH_deltaNB',ethnic_dca,10,'Spearman-pruned','Cox PH',delta=True)


In [ ]:
# ethnicity_10y_Spearman-pruned_Elastic-net_Cox_calibration
run('ethnicity_10y_Spearman-pruned_Elastic-net_Cox_calibration',ethnic_cal,10,'Spearman-pruned','Elastic-net Cox')


In [ ]:
# ethnicity_10y_Spearman-pruned_Elastic-net_Cox_DCA
run('ethnicity_10y_Spearman-pruned_Elastic-net_Cox_DCA',ethnic_dca,10,'Spearman-pruned','Elastic-net Cox')


In [ ]:
# ethnicity_10y_Spearman-pruned_Elastic-net_Cox_deltaNB
run('ethnicity_10y_Spearman-pruned_Elastic-net_Cox_deltaNB',ethnic_dca,10,'Spearman-pruned','Elastic-net Cox',delta=True)


In [ ]:
# ethnicity_10y_Spearman-pruned_XGBoost_survival_calibration
run('ethnicity_10y_Spearman-pruned_XGBoost_survival_calibration',ethnic_cal,10,'Spearman-pruned','XGBoost survival')


In [ ]:
# ethnicity_10y_Spearman-pruned_XGBoost_survival_DCA
run('ethnicity_10y_Spearman-pruned_XGBoost_survival_DCA',ethnic_dca,10,'Spearman-pruned','XGBoost survival')


In [ ]:
# ethnicity_10y_Spearman-pruned_XGBoost_survival_deltaNB
run('ethnicity_10y_Spearman-pruned_XGBoost_survival_deltaNB',ethnic_dca,10,'Spearman-pruned','XGBoost survival',delta=True)


In [ ]:
# ethnicity_10y_Spearman-pruned_DeepSurv_calibration
run('ethnicity_10y_Spearman-pruned_DeepSurv_calibration',ethnic_cal,10,'Spearman-pruned','DeepSurv')


In [ ]:
# ethnicity_10y_Spearman-pruned_DeepSurv_DCA
run('ethnicity_10y_Spearman-pruned_DeepSurv_DCA',ethnic_dca,10,'Spearman-pruned','DeepSurv')


In [ ]:
# ethnicity_10y_Spearman-pruned_DeepSurv_deltaNB
run('ethnicity_10y_Spearman-pruned_DeepSurv_deltaNB',ethnic_dca,10,'Spearman-pruned','DeepSurv',delta=True)


In [ ]:
# ethnicity_10y_Raw106_Cox_PH_calibration
run('ethnicity_10y_Raw106_Cox_PH_calibration',ethnic_cal,10,'Raw106','Cox PH')


In [ ]:
# ethnicity_10y_Raw106_Cox_PH_DCA
run('ethnicity_10y_Raw106_Cox_PH_DCA',ethnic_dca,10,'Raw106','Cox PH')


In [ ]:
# ethnicity_10y_Raw106_Cox_PH_deltaNB
run('ethnicity_10y_Raw106_Cox_PH_deltaNB',ethnic_dca,10,'Raw106','Cox PH',delta=True)


In [ ]:
# ethnicity_10y_Raw106_Elastic-net_Cox_calibration
run('ethnicity_10y_Raw106_Elastic-net_Cox_calibration',ethnic_cal,10,'Raw106','Elastic-net Cox')


In [ ]:
# ethnicity_10y_Raw106_Elastic-net_Cox_DCA
run('ethnicity_10y_Raw106_Elastic-net_Cox_DCA',ethnic_dca,10,'Raw106','Elastic-net Cox')


In [ ]:
# ethnicity_10y_Raw106_Elastic-net_Cox_deltaNB
run('ethnicity_10y_Raw106_Elastic-net_Cox_deltaNB',ethnic_dca,10,'Raw106','Elastic-net Cox',delta=True)


In [ ]:
# ethnicity_10y_Raw106_XGBoost_survival_calibration
run('ethnicity_10y_Raw106_XGBoost_survival_calibration',ethnic_cal,10,'Raw106','XGBoost survival')


In [ ]:
# ethnicity_10y_Raw106_XGBoost_survival_DCA
run('ethnicity_10y_Raw106_XGBoost_survival_DCA',ethnic_dca,10,'Raw106','XGBoost survival')


In [ ]:
# ethnicity_10y_Raw106_XGBoost_survival_deltaNB
run('ethnicity_10y_Raw106_XGBoost_survival_deltaNB',ethnic_dca,10,'Raw106','XGBoost survival',delta=True)


In [ ]:
# ethnicity_10y_Raw106_DeepSurv_calibration
run('ethnicity_10y_Raw106_DeepSurv_calibration',ethnic_cal,10,'Raw106','DeepSurv')


In [ ]:
# ethnicity_10y_Raw106_DeepSurv_DCA
run('ethnicity_10y_Raw106_DeepSurv_DCA',ethnic_dca,10,'Raw106','DeepSurv')


In [ ]:
# ethnicity_10y_Raw106_DeepSurv_deltaNB
run('ethnicity_10y_Raw106_DeepSurv_deltaNB',ethnic_dca,10,'Raw106','DeepSurv',delta=True)


In [ ]:
# Core models: Score249, 5 years, Uno_C
run("ethnicity_core_Score249_5y_Uno_C",ethnic_core,5,'Score249','Uno_C')


In [ ]:
# Core models: Score249, 5 years, IBS
run("ethnicity_core_Score249_5y_IBS",ethnic_core,5,'Score249','IBS')


In [ ]:
# Core models: Score106, 5 years, Uno_C
run("ethnicity_core_Score106_5y_Uno_C",ethnic_core,5,'Score106','Uno_C')


In [ ]:
# Core models: Score106, 5 years, IBS
run("ethnicity_core_Score106_5y_IBS",ethnic_core,5,'Score106','IBS')


In [ ]:
# Core models: Score249, 10 years, Uno_C
run("ethnicity_core_Score249_10y_Uno_C",ethnic_core,10,'Score249','Uno_C')


In [ ]:
# Core models: Score249, 10 years, IBS
run("ethnicity_core_Score249_10y_IBS",ethnic_core,10,'Score249','IBS')


In [ ]:
# Core models: Score106, 10 years, Uno_C
run("ethnicity_core_Score106_10y_Uno_C",ethnic_core,10,'Score106','Uno_C')


In [ ]:
# Core models: Score106, 10 years, IBS
run("ethnicity_core_Score106_10y_IBS",ethnic_core,10,'Score106','IBS')


In [ ]:
# Report figure generation status.
status=pd.DataFrame(LOG)
print(status.groupby('Status').size())
print(status[status.Status!='saved'].to_string(index=False))
print('Figures:',FIG)
print('Source tables:',TABLE)
